# Paso 1 · ¿El agente decide bien cuándo buscar? ¿Cuánto pesan los resultados?

En este primer paso queremos responder a tres preguntas:

1. **¿El agente se comporta bien?** Es decir, ¿usa la búsqueda web cuando la necesita y **no** la usa cuando no hace falta?
2. **¿Cómo accedemos a los `ToolMessage`?** (los mensajes que contienen lo que devolvió Tavily)
3. **¿Cuántos tokens gasta el agente y cuánto pesan los `ToolMessage`?**

> El código está en `src/espadas_langgraph_websearchagent/` (`searchagent.py` y `tokens.py`). Aquí solo lo usamos y explicamos los resultados.

## 0. Configuración

In [ ]:
# Cargamos las claves de API desde el fichero .env (ANTHROPIC_API_KEY y TAVILY_API_KEY).
# Si no están, las pedimos por teclado.
import getpass
import os

from dotenv import load_dotenv

load_dotenv()

if not os.environ.get("ANTHROPIC_API_KEY"):
    os.environ["ANTHROPIC_API_KEY"] = getpass.getpass("Anthropic API key:\n")

if not os.environ.get("TAVILY_API_KEY"):
    os.environ["TAVILY_API_KEY"] = getpass.getpass("Tavily API key:\n")

## 1. El agente

El agente es un grafo de LangGraph con dos nodos:

```
START → llm_call ──(¿pide tool?)── sí ──→ tool_node ──┐
            ▲  │                                      │
            │  no → END                               │
            └─────────────────────────────────────────┘
```

- `llm_call`: Claude Haiku 4.5 lee la conversación y decide si responde o si pide una búsqueda.
- `tool_node`: ejecuta la búsqueda en Tavily y devuelve el resultado como `ToolMessage`.

### ¿Por qué la fecha va en el system prompt y no en una tool?

El modelo no sabe qué día es hoy. Si le diéramos una tool `get_date()`:
- tendría que hacer **una llamada extra** al LLM (pedir la fecha → recibirla → seguir), y en cada llamada se reenvía toda la conversación;
- además, la definición de la tool ocuparía tokens en **todas** las llamadas.

Meter la fecha en el system prompt cuesta unos 10 tokens y no añade ninguna llamada. También metemos la **fecha de corte** (*knowledge cutoff*) para que el modelo sepa qué cosas pueden haber cambiado desde que lo entrenaron.

In [ ]:
from espadas_langgraph_websearchagent import searchagent

agent = searchagent.create_agent_with_search_tool()

# Dibujamos el grafo
searchagent.show_graph(agent)

# Así queda el system prompt con la fecha de hoy y la fecha de corte
print(searchagent.build_system_prompt())

## 2. Una pregunta de ejemplo

Hacemos una pregunta que **sí** necesita búsqueda (la última versión de Python cambia con el tiempo) y vemos todos los mensajes de la conversación.

In [ ]:
result = searchagent.run_agent_with_search_tool(
    agent, "¿Cuál es la última versión de Python?", draw_graph=False
)

## 3. Acceso a los `ToolMessage`

`agent.invoke(...)` devuelve un diccionario con la lista `messages`:

| Tipo | Quién lo crea | Qué contiene |
|---|---|---|
| `HumanMessage` | el usuario | la pregunta |
| `AIMessage` (con `tool_calls`) | el LLM | la búsqueda que quiere hacer (query, parámetros) |
| `ToolMessage` | nuestro `tool_node` | el resultado de Tavily |
| `AIMessage` (final) | el LLM | la respuesta |

Cada `ToolMessage` tiene dos campos importantes:
- **`content`**: el texto que **se envía al LLM**. Es lo que cuesta tokens.
- **`artifact`**: el resultado original de Tavily (un `dict`). **No se envía al LLM**. Lo guardamos para poder analizarlo.

In [ ]:
tool_messages = searchagent.get_tool_messages(result)
print(f"Número de ToolMessage: {len(tool_messages)}")

for tm in tool_messages:
    print("\n--- ToolMessage ---")
    print("tool_call_id:", tm.tool_call_id)
    print("Tipo del artifact:", type(tm.artifact).__name__)
    if isinstance(tm.artifact, dict):
        print("Claves del artifact:", list(tm.artifact.keys()))
    print(f"Longitud del content: {len(tm.content):,} caracteres")
    print("Principio del content (lo que lee el LLM):")
    print(tm.content[:500], "...")

# Los argumentos con los que el LLM llamó a la tool
print("\nBúsquedas pedidas por el LLM:", searchagent.get_search_queries(result))

Fíjate en que el `content` empieza por `{'query': ..., 'follow_up_questions': None, 'answer': None, 'images': [], ...`.
LangChain convierte el `dict` de Tavily a texto con `str()`, así que el LLM recibe **todas** las claves, aunque muchas estén vacías. Lo aprovecharemos en el Paso 3.

## 4. ¿Cómo se calculan los tokens de los `ToolMessage`? (explicación detallada)

Anthropic no nos dice "el ToolMessage ocupó X tokens". En cada llamada al LLM solo nos dice:
- `input_tokens`: cuántos tokens **entraron**,
- `output_tokens`: cuántos tokens **salieron**.

LangChain lo guarda en `AIMessage.usage_metadata`.

**La API no tiene memoria**: en cada llamada se reenvía la conversación entera. Cuando el agente busca, hace **2 llamadas** al LLM:

```
Llamada 1 (el LLM decide buscar)
  ENTRA:  [system + tools] + [pregunta]                                    = input_1
  SALE:   [AIMessage con la tool_call]                                     = output_1

  ... Tavily busca y creamos el ToolMessage (no es una llamada al LLM) ...

Llamada 2 (el LLM responde)
  ENTRA:  [system + tools] + [pregunta] + [AIMessage tool_call] + [ToolMessage]   = input_2
  SALE:   [respuesta final]                                                       = output_2
```

La entrada de la llamada 2 es la de la llamada 1, más lo que escribió el LLM, más el ToolMessage:

$$\text{input}_2 = \text{input}_1 + \text{output}_1 + \text{tokens}_{ToolMessage}$$

y despejando:

$$\boxed{\text{tokens}_{ToolMessage} = \text{input}_2 - \text{input}_1 - \text{output}_1}$$

### Diferencia entre la `tool_call` y el `ToolMessage`

| | `tool_call` | `ToolMessage` |
|---|---|---|
| Quién lo escribe | el LLM | Tavily (a través de nuestro código) |
| Tipo de token | **salida** (output) de la llamada 1 | **entrada** (input) de la llamada 2 |
| Tamaño típico | ~50-150 tokens | ~2.500-5.000 tokens |
| Precio Haiku 4.5 | 5 $ / millón | 1 $ / millón |

### Detalles a tener en cuenta
1. Si el LLM pide **varias búsquedas a la vez**, la resta da el total de todos los ToolMessage de esa vuelta.
2. El `AIMessage` con la `tool_call` cuenta como salida en la llamada 1 y como entrada en la llamada 2; el formato puede variar en unos pocos tokens.
3. Cada `ToolMessage` va dentro de un bloque `tool_result` con algún token de "envoltorio", que este método cuenta como parte del ToolMessage.
4. En un chatbot con historial, el ToolMessage **se vuelve a pagar en cada pregunta siguiente**, porque se reenvía con toda la conversación.

In [ ]:
from espadas_langgraph_websearchagent.tokens import get_token_usage, print_token_usage

usage = get_token_usage(result)
print_token_usage(usage)

In [ ]:
# Hacemos la resta "a mano" para verla paso a paso
calls = usage["calls"]
if len(calls) >= 2:
    input_1 = calls[0]["input_tokens"]
    output_1 = calls[0]["output_tokens"]
    input_2 = calls[1]["input_tokens"]
    print(f"input_1  (system + tools + pregunta)               = {input_1:>6,}")
    print(f"output_1 (tool_call escrita por el LLM)            = {output_1:>6,}")
    print(f"input_2  (todo lo anterior + ToolMessage)          = {input_2:>6,}")
    print(f"ToolMessage = {input_2:,} - {input_1:,} - {output_1:,} = {input_2 - input_1 - output_1:,} tokens")
    print(f"Peso del ToolMessage en el total: {(input_2 - input_1 - output_1) / usage['total_tokens']:.0%}")
else:
    print("El agente no buscó: no hay ToolMessage.")

### Comprobación con el método "exacto": la API `count_tokens`

Anthropic tiene una API **gratuita** que cuenta los tokens de una conversación sin ejecutarla. La usamos para comprobar la resta:

1. contamos la conversación **con** los resultados de Tavily,
2. contamos la misma conversación con el contenido de cada ToolMessage sustituido por `"-"`,
3. la diferencia son los tokens del contenido de los ToolMessage.

Si los dos métodos dan casi lo mismo, la resta es fiable.

In [ ]:
from espadas_langgraph_websearchagent.tokens import count_tool_message_tokens_exact

model = searchagent.create_model()
tools = [searchagent.create_default_search_tool()]

exact = count_tool_message_tokens_exact(model, tools, searchagent.build_system_prompt(), result)
estimated = usage["tool_result_tokens"]

print(f"Método de la resta (usage_metadata): {estimated:,} tokens")
print(f"Método count_tokens (exacto):        {exact:,} tokens")
print(f"Diferencia:                          {estimated - exact:,} tokens (envoltorio tool_result + formato)")

### ¿De dónde salen los tokens de la primera llamada?

Incluso cuando el agente **no busca**, la primera llamada ya tiene unos 2.000 tokens de entrada. Vamos a ver de dónde salen:
- el system prompt,
- la definición de la tool (el JSON con nombre, descripción y parámetros),
- un **system prompt oculto** que Anthropic añade siempre que hay tools (en Claude Haiku 4.5 son **496 tokens**, según la documentación de Anthropic),
- la pregunta.

In [ ]:
from espadas_langgraph_websearchagent.tokens import count_text_tokens, count_tool_definition_tokens

tool_tokens = count_tool_definition_tokens(model, tools)
system_tokens = count_text_tokens(model, searchagent.build_system_prompt())

print(f"System prompt:                         {system_tokens:>6,} tokens")
print(f"Tools (JSON + prompt oculto de 496):   {tool_tokens:>6,} tokens")
print(f"   de ellos, el JSON de Tavily ≈       {tool_tokens - 496:>6,} tokens")
print(f"Entrada real de la 1ª llamada:         {usage['first_call_input_tokens']:>6,} tokens")

## 5. Evaluación con la batería de preguntas

Pasamos las 9 preguntas de `questions.py`. Cada una dice si el agente **debería** buscar o no. Medimos:
- si la decisión es correcta,
- los tokens totales, los de los ToolMessage y el coste.

In [ ]:
from espadas_langgraph_websearchagent.evaluation import evaluate_agent, print_summaries, summarize

rows = evaluate_agent(agent)
print()
print_summaries([summarize("0_base", rows)])

In [ ]:
# Peso de los ToolMessage en las preguntas en las que buscó
print("Pregunta                                         | total  | ToolMessage | peso")
for row in rows:
    if row["busco"]:
        share = row["tokens_toolmessage"] / row["tokens_totales"]
        print(f"{row['pregunta'][:48]:<48} | {row['tokens_totales']:>6,} | {row['tokens_toolmessage']:>11,} | {share:.0%}")

In [ ]:
# ¿Qué respondió en las preguntas en las que se equivocó? ¿Se inventó algo?
for row in rows:
    if not row["decision_ok"]:
        print("=" * 80)
        print("PREGUNTA:", row["pregunta"])
        print("Debía buscar:", row["debia_buscar"], "| Buscó:", row["busco"])
        print(row["respuesta"])

## 6. Conclusiones del Paso 1

> ⚠️ Los números de abajo son de la **primera ejecución** del proyecto (`testing_agent.ipynb`). Al ejecutar este notebook, actualízalos con los tuyos.

- **Comportamiento:** el agente acertó **7 de 9** decisiones. Falló en:
  - *"¿Quién ganó el Mundial de 2026?"*: no buscó, aunque el Mundial 2026 es posterior a su fecha de corte (febrero 2025).
  - *"¿Qué tiempo hace hoy en Sevilla?"*: no buscó (los modelos suelen responder "no tengo acceso a datos en tiempo real").
- **Coste fijo:** sin buscar, cada pregunta cuesta unos **2.400-2.800 tokens**. Casi todo es la definición de las tools, que se envía siempre.
- **Coste con búsqueda:** sube a **~7.900-8.400 tokens**. Los `ToolMessage` ocupan **~2.500-3.000 tokens (32-37% del total)**.
- **Cuando busca, el coste fijo se paga DOS veces**, porque hay dos llamadas al LLM y en las dos van el system prompt y las tools.
- **Búsquedas en paralelo:** con *"¿Cuántos CV tiene la Honda Rebel? busca nuevos modelos"* el LLM lanzó **2 búsquedas `advanced` a la vez**: 4.741 tokens de ToolMessage (**45% del total**), y 2 créditos de Tavily por búsqueda.
- La resta `input_2 - input_1 - output_1` coincide casi exactamente con `count_tokens`. La diferencia son unos pocos tokens del envoltorio `tool_result`.